[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/23_cross_attention.ipynb)

# 🟠 Medium: Multi-Head Cross-Attention

Implement **multi-head cross-attention** (encoder-decoder attention).

### Signature
```python
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x_q: Tensor, x_kv: Tensor) -> Tensor:
        # x_q: (B, S_q, D) — decoder queries
        # x_kv: (B, S_kv, D) — encoder keys/values
```

### Key Differences from Self-Attention
- Q comes from the decoder, K and V come from the encoder
- No causal mask (all encoder positions visible)

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import torch.nn as nn
import math

In [64]:
# ✏️ YOUR IMPLEMENTATION HERE

class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        # pass  # W_q, W_k, W_v, W_o
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        # weight matrices for query, key, value, and output projections
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        
        # # Inefficient alternative
        # self.heads = nn.ModuleList([nn.Linear(d_model, head_dim) for _ in range(num_heads)])

    def forward(self, x_q, x_kv):
        # pass  # Q from x_q, K/V from x_kv, no causal mask
        batch_size, seq_len_q = x_q.size(0), x_q.size(1)
        seq_len_kv = x_kv.size(1)

        Q = self.W_q(x_q)
        K = self.W_k(x_kv)
        V = self.W_v(x_kv)
        print(Q.shape, K.shape, V.shape)
        
        # [Batch, Seq_len, d_model] -> [Batch, Seq_len, num_heads, head_dim]
        Q_split = Q.view(batch_size, seq_len_q, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        K_split = K.view(batch_size, seq_len_kv, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        V_split = V.view(batch_size, seq_len_kv, self.num_heads, self.head_dim).permute(0, 2, 1, 3)

        print(Q_split.shape, K_split.shape, V_split.shape)

        multi_head_attn = torch.softmax(Q_split @ K_split.transpose(-2, -1) / math.sqrt(self.head_dim), dim=-1) @ V_split
        print(multi_head_attn.shape) # the seq length of the output is the same as the query sequence length
        attn = multi_head_attn.permute(0, 2, 1, 3).contiguous().view(batch_size, seq_len_q, self.d_model) # concatenate the heads
        print(attn.shape)
        return self.W_o(attn)

In [65]:
# 🧪 Debug
attn = MultiHeadCrossAttention(64, 4)
x_q = torch.randn(2, 6, 64)
x_kv = torch.randn(2, 10, 64)
print('Output:', attn(x_q, x_kv).shape)

torch.Size([2, 6, 64]) torch.Size([2, 10, 64]) torch.Size([2, 10, 64])
torch.Size([2, 4, 6, 16]) torch.Size([2, 4, 10, 16]) torch.Size([2, 4, 10, 16])
torch.Size([2, 4, 6, 16])
torch.Size([2, 6, 64])
Output: torch.Size([2, 6, 64])


In [66]:
# ✅ SUBMIT
from torch_judge import check
check('cross_attention')


🧪 Testing: Multi-Head Cross-Attention (Medium)
──────────────────────────────────────────────────
torch.Size([2, 6, 64]) torch.Size([2, 10, 64]) torch.Size([2, 10, 64])
torch.Size([2, 4, 6, 16]) torch.Size([2, 4, 10, 16]) torch.Size([2, 4, 10, 16])
torch.Size([2, 4, 6, 16])
torch.Size([2, 6, 64])
  ✅ [1/4] Output shape (1.2ms)
torch.Size([1, 3, 32]) torch.Size([1, 20, 32]) torch.Size([1, 20, 32])
torch.Size([1, 2, 3, 16]) torch.Size([1, 2, 20, 16]) torch.Size([1, 2, 20, 16])
torch.Size([1, 2, 3, 16])
torch.Size([1, 3, 32])
  ✅ [2/4] Q and KV different lengths (0.4ms)
torch.Size([1, 4, 32]) torch.Size([1, 6, 32]) torch.Size([1, 6, 32])
torch.Size([1, 2, 4, 16]) torch.Size([1, 2, 6, 16]) torch.Size([1, 2, 6, 16])
torch.Size([1, 2, 4, 16])
torch.Size([1, 4, 32])
torch.Size([1, 4, 32]) torch.Size([1, 6, 32]) torch.Size([1, 6, 32])
torch.Size([1, 2, 4, 16]) torch.Size([1, 2, 6, 16]) torch.Size([1, 2, 6, 16])
torch.Size([1, 2, 4, 16])
torch.Size([1, 4, 32])
  ✅ [3/4] No causal mask — all KV